# Uji cross-dataset dan probe klaim pendek (TF-IDF + Logistic Rregression)

Memanggil `classification/cross_dataset.py` dan `classification/short_claims_probe.py`

Dataset Mafindo dari Hugging Face

File yang digunakan:
- `cross_dataset.py`, `short_claims_probe.py` dan `mendeley_audit.py`
- `600 news with valid hoax label.csv` dan `600 news labelling process.xlsx` untuk uji cross-dataset
- `komdigi_hoaks.csv` dari Kaggle ireddragonicy/indonesian-hoax-news-dataset untuk probe klaim pendek
- `no_match_queries.jsonl` (opsional) untuk cek kueri no_match di langkah 5


## 1. Setup

In [1]:
!pip -q install datasets openpyxl

## 2. Connect Google Drive dan folder

In [8]:
import os, shutil
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/SEMESTER 7/NLP/Tubes"

dest = {
    "cross_dataset.py": "classification",
    "short_claims_probe.py": "classification",
    "mendeley_audit.py": "classification/data_audit",
    "600 news with valid hoax label.csv": "data/raw/mendeley",
    "600 news labelling process.xlsx": "data/raw/mendeley",
    "komdigi_hoaks.csv": "data/raw/refuted_claims",
    "no_match_queries.jsonl": "data/interim/classifier",
}
print("Isi folder Drive:", os.listdir(DRIVE_DIR))
for name, folder in dest.items():
    os.makedirs(folder, exist_ok=True)
    src = os.path.join(DRIVE_DIR, name)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(folder, name))

missing = [os.path.join(f, k) for k, f in dest.items() if not os.path.exists(os.path.join(f, k))]
print("Belum ada:" if missing else "Semua berkas siap.", missing or "")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Isi folder Drive: ['600 news labelling process.xlsx', '600 news with valid hoax label.csv', 'mendeley_audit.py', 'cross_dataset.py', 'short_claims_probe.py', 'Copy of komdigi_hoaks.csv']
Semua berkas siap. 


## 3. Uji cross-dataset

In [4]:
!python classification/cross_dataset.py --data-dir data/raw/mendeley

README.md: 100% 858/858 [00:00<00:00, 3.66MB/s]

train.csv: downloading bytes:  33% 11.5M/35.1M [00:00<00:01, 16.6MB/s,  245kB/s  ]
train.csv: downloading bytes: 100% 20.9M/20.9M [00:01<00:00, 19.5MB/s, 2.03MB/s  ]
train.csv: reconstructing file: 100% 35.1M/35.1M [00:01<00:00, 32.8MB/s, 3.43MB/s  ]

test.csv: downloading bytes:  43% 3.74M/8.61M [00:00<00:00, 6.09MB/s, 15.0kB/s  ]
test.csv: downloading bytes: 100% 5.14M/5.14M [00:00<00:00, 5.62MB/s,  498kB/s  ]
test.csv: reconstructing file: 100% 8.61M/8.61M [00:00<00:00, 9.42MB/s,  837kB/s  ]
Generating train split: 100% 23641/23641 [00:00<00:00, 53080.49 examples/s]
Generating test split: 100% 5911/5911 [00:00<00:00, 50062.86 examples/s]
Mafindo train 23641 (hoax 0.473) | test 5911 (hoax 0.486) | Mendeley 600 (hoax 0.383; tanpa bantahan 280)
A_mafindo_to_mendeley                      macroF1 0.569 CI[0.529, 0.612] | AUROC 0.597 CI[0.552, 0.644] | hoax P 0.50 R 0.36 | pred/true hoax 0.28/0.38 | n=600
B_mafindo_to_mendeley_no_debunk    

## 4. Probe klaim pendek
Membangun 300 klaim (150 judul Komdigi tanpa prefiks, 150 kalimat pertama artikel valid Mafindo test) lalu menilai model Mafindo

In [9]:
!python classification/short_claims_probe.py --komdigi-csv data/raw/refuted_claims/komdigi_hoaks.csv

300 klaim (150 hoaks) | komdigi rows 3048 | seed 13
mafindo TF-IDF+LR -> klaim pendek: macroF1 0.556 CI[0.498, 0.613] | AUROC 0.962 CI[0.938, 0.979] | hoax P 0.57 R 1.00 | pred/true hoax 0.88/0.50
AUROC panjang kata saja: 0.03 | rata-rata kata per kelas: {'hoax': 8.4, 'valid': 19.9}

Contoh (3 per kelas) untuk cek mata:
  [hoax] Aceh Kini Kibarkan Bendera Sendiri
  [hoax] WNI dengan Tabungan Lebih Dari Rp3 Miliar Wajib Beli Obligasi Merah Putih
  [hoax] Anies Baswedan Janji Gratiskan BBM jika Terpilih pada Pilpres 2029
  [valid] Ketua Fraksi PAN DPR RI Saleh P Daulay bertanya-tanya soal kritik dari Sekjen PDIP Hasto Kristiyanto terhadap permintaan maaf dari Presiden Joko Widodo (Jokowi).
  [valid] Ketua Umum PKB Muhaimin Iskandar atau Cak Imin sempat bertemu Ketum Gerindra Prabowo Subianto untuk bicara berbagai hal, termasuk Pilkada Jakarta 2024.
  [valid] Wali Kota Tangerang Selatan (Tangsel) Benyamin Davnie resmi menjadi kader Partai Gerindra.

klaim -> data/interim/classifier/short_

## 5. Cek kueri no_match terhadap basis Komdigi
Menandai kueri yang terlalu mirip dengan judul Komdigi dan Penanda untuk ditinjau manual

In [11]:
!python classification/no_match_check.py --queries data/interim/classifier/no_match_queries.jsonl --komdigi-csv data/raw/refuted_claims/komdigi_hoaks.csv

144 kueri vs 3048 judul Komdigi | cosine maks: median 0.226, p95 0.351, maks 0.431
Ditandai (>= 0.5): 0
  sc265 0.431
    kueri : Khofifah Indar Parawansa dan Emil Dardak telah mengantongi dukungan enam parpol di Pilgub Jatim 2024.
    judul : Gubernur Khofifah Indar Parawansa Umumkan Dana Hibah dari Arab Saudi
  sc169 0.429
    kueri : Polisi buru seorang pria paruh baya yang diduga melakukan pelecehan seksual terhadap wartawan perempuan di Kot
    judul : Video Guru di Kabupaten Grobogan Lakukan Pelecehan Seksual terhadap Murid
  sc254 0.400
    kueri : Menteri Investasi Bahlil Lahadalia disebut masuk bursa caketum Partai Golkar.
    judul : Prabowo Pecat Menteri ESDM Bahlil Lahadalia
  sc290 0.381
    kueri : Presiden Joko Widodo atau Jokowi meresmikan layanan digital untuk mempermudah pemohon dalam mengurus perizinan
    judul : Sertifikat Digital Mempermudah Pemerintah Merampas Tanah Masyarakat
  sc252 0.372
    kueri : Imam Besar Masjid Istiqlal Nasaruddin Umar mengungkapkan, Pau

## 6. Download hasil

In [12]:
import shutil
from google.colab import files
os.makedirs("out", exist_ok=True)
for f in ["results/cross_dataset.json", "results/short_claims_probe.json", "results/no_match_check.json",
          "data/manifests/short_claims_manifest.csv", "data/interim/classifier/short_claims.jsonl"]:
    if os.path.exists(f):
        shutil.copy(f, "out/")
shutil.make_archive("hasil", "zip", "out")
files.download("hasil.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>